In [5]:
import numpy as np
import re
import os

def restore_screen_data(filename):
    #Read header.
    with open(filename, "r") as f:
        header = f.readline().strip()

    #Get num of cycles.
    match = re.search(r"num\. of cycles:\s*(\d+)", header)
    
    if not match:
        raise ValueError("Can't find number of cycles.")

    num_of_cycles = int(match.group(1))
    print(f"Number oif cycles: {num_of_cycles}")

    #Load the data without the header.
    data = np.loadtxt(filename, delimiter="\t", skiprows=1)

    total_rows = data.shape[0]

    if total_rows % num_of_cycles != 0:
        raise ValueError("Number of raws can not be divided with the number of cycles.")

    M = total_rows // num_of_cycles

    print(f"Restored Python list shape: ({num_of_cycles}, 3, {M})")

    #3D Python list
    restored_3d = data.reshape(num_of_cycles, 3, M)

    #Restore the strucutre.
    restored_results = [
        [
            restored_3d[i, 0].copy(),
            restored_3d[i, 1].copy(),
            restored_3d[i, 2].copy()
        ]
        for i in range(num_of_cycles)
    ]

    #Save with vstack.
    stacked_cycles = [
        np.column_stack((t, ch1, ch2))
        for t, ch1, ch2 in restored_results
    ]

    final_data = np.vstack(stacked_cycles)

    base, ext = os.path.splitext(filename)
    new_filename = base + "_restored" + ext

    header_out = f"time[s]\tCH1[V]\tCH2[V] (num. of cycles: {num_of_cycles})"

    np.savetxt(
        new_filename,
        final_data,
        delimiter="\t",
        header=header_out,
        comments=""
    )

    return restored_results

#----------EXAMPLE-----------

#The row below restores the data saved by "read_screen_data" in a 3D python list and also saves it in a correct nx3 column stack with "<filename>_restored.txt"

#my_restored_data=restore_screen_data("task_01_data.txt")
#import matplotlib.pyplot as plt
#plt.plot(my_restored_data[0][0], my_restored_data[0][1])